## Creating new columns for raw counts and percentages and filling in the missing values

In [57]:
import pandas as pd

In [60]:
# df1 = pd.read_excel('K13_validated_v1.xlsx')
df2 = pd.read_excel('K13_non_validated.xlsx')

In [61]:
df3 = pd.read_excel('K13_coordinates_v1.xlsx')

In [5]:
# df4 = pd.read_excel('kilinga2_coordinates.xlsx')

In [68]:
df2.head()

,pmid,year_pub,year_sample,country,region/province,town,participants_age,participants_age_unit,sample_size,first_line_act,genotyping_assay,study_population,samples_genotyped,where_sequenced,other_k13_mutations,Frequency,data_extraction_comments,journal_access,raw_count,percentage_frequency
0,34551228,2021,2015,Uganda,Northern Uganda,Gulu,≥0.5,years,77,AL,Sanger,symptomatic,77,missing,C532S,2 (2.6),raw counts,open,2,2.6
1,34551228,2021,2015,Uganda,Northern Uganda,Gulu,≥0.5,years,77,AL,Sanger,symptomatic,77,missing,V661I,1 (1.3),raw counts,open,1,1.3
2,34551228,2021,2016,Uganda,Northern Uganda,Gulu,≥0.5,years,94,AL,Sanger,symptomatic,94,missing,G595C,1 (1.1),raw counts,open,1,1.1
3,34551228,2021,2018,Uganda,Northern Uganda,Gulu,≥0.5,years,67,AL,Sanger,symptomatic,67,missing,A578S,1 (1.5),raw counts,open,1,1.5
4,34551228,2021,2018,Uganda,Northern Uganda,Gulu,≥0.5,years,67,AL,Sanger,symptomatic,67,missing,S695T,1 (1.5),raw counts,open,1,1.5


In [29]:
# remove spaces in column names and replace with  "-"
df1.columns = df1.columns.str.strip().str.lower().str.replace(" ", "_")

In [30]:
print(df1.columns.tolist())

['pmid', 'year_pub', 'year_sample', 'country', 'region/province', 'town', 'participants_age', 'participants_age_unit', 'sample_size', 'first_line_act', 'genotyping_assay', 'study_population', 'samples_genotyped', 'k13_validated_mutations', 'type_of_mutation', 'mutation_frequency_(%)', 'where_sequenced', 'other_k13_mutations', 'data_extraction_comments', 'journal_access']


#### Create the raw_counts and percentage column for better analysis

In [67]:
import pandas as pd
import re
import numpy as np

def extract_values(row):
    freq = str(row["Frequency"]).strip()
    comment = str(row["data_extraction_comments"]).strip().upper()

    # Case 1: Direct percentage
    if "PERCENTAGE" in comment:
        try:
            return pd.Series({
                "raw_count": None,
                "percentage_frequency": float(freq)
            })
        except ValueError:
            return pd.Series({
                "raw_count": None,
                "percentage_frequency": None
            })

    # Case 2: RAW COUNTS
    if "RAW COUNTS" in comment:
        # Case 2a: format like "3 (3.4)" or "1(6.7)"
        match = re.search(r"^(\d+)\s*\(([\d\.]+)\)$", freq)
        if match:
            raw = match.group(1)
            percentage = float(match.group(2))
            return pd.Series({
                "raw_count": raw,
                "percentage_frequency": percentage
            })

        # Case 2b: format like "x/y"
        if "/" in freq:
            try:
                num, denom = map(float, freq.split("/"))
                percentage = round((num / denom) * 100, 2) if denom else None
                return pd.Series({
                    "raw_count": freq,
                    "percentage_frequency": percentage
                })
            except:
                return pd.Series({
                    "raw_count": freq,
                    "percentage_frequency": None
                })

        # Case 2c: just a number like "5"
        if freq.isdigit():
            try:
                raw = float(freq)
                total = float(row.get("samples_genotyped", np.nan))
                percentage = round((raw / total) * 100, 2) if pd.notna(total) and total > 0 else None
                return pd.Series({
                    "raw_count": freq,
                    "percentage_frequency": percentage
                })
            except:
                return pd.Series({
                    "raw_count": freq,
                    "percentage_frequency": None
                })
        else:
            return pd.Series({
                "raw_count": freq,
                "percentage_frequency": None
            })

    # Fallback
    return pd.Series({
        "raw_count": None,
        "percentage_frequency": None
    })
df2[["raw_count", "percentage_frequency"]] = df2.apply(extract_values, axis=1)

In [69]:
import pandas as pd
import numpy as np

# Make sure raw_count and percentage_frequency columns exist and are strings
df2["raw_count"] = df2["raw_count"].astype(str)
df2["percentage_frequency"] = df2.get("percentage_frequency", pd.Series(np.nan, index=df1.index))

# Only update where percentage_frequency is missing
mask_missing = df2["percentage_frequency"].isna()

# Extract percentage from raw_count where it's missing
extracted_pct = df2.loc[mask_missing, "raw_count"].str.extract(r"\(([\d\.]+)\)")
df2.loc[mask_missing, "percentage_frequency"] = pd.to_numeric(extracted_pct[0], errors="coerce")

# Strip the bracketed part from raw_count to keep only the main value
df2["raw_count"] = df2["raw_count"].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()

In [70]:
# Set percentage to 0 where mutation_frequency is '0'
df2.loc[df2["Frequency"].str.strip() == "0", "Frequency"] = 0.0

In [71]:
# Keep only the part before the slash if it's in x/y format
df2["raw_count"] = df2["raw_count"].astype(str).str.extract(r"^(\d+)")

In [72]:
df2.head(20)

,pmid,year_pub,year_sample,country,region/province,town,participants_age,participants_age_unit,sample_size,first_line_act,genotyping_assay,study_population,samples_genotyped,where_sequenced,other_k13_mutations,Frequency,data_extraction_comments,journal_access,raw_count,percentage_frequency
0,34551228,2021,2015,Uganda,Northern Uganda,Gulu,≥0.5,years,77,AL,Sanger,symptomatic,77,missing,C532S,2 (2.6),raw counts,open,2,2.60
1,34551228,2021,2015,Uganda,Northern Uganda,Gulu,≥0.5,years,77,AL,Sanger,symptomatic,77,missing,V661I,1 (1.3),raw counts,open,1,1.30
2,34551228,2021,2016,Uganda,Northern Uganda,Gulu,≥0.5,years,94,AL,Sanger,symptomatic,94,missing,G595C,1 (1.1),raw counts,open,1,1.10
3,34551228,2021,2018,Uganda,Northern Uganda,Gulu,≥0.5,years,67,AL,Sanger,symptomatic,67,missing,A578S,1 (1.5),raw counts,open,1,1.50
4,34551228,2021,2018,Uganda,Northern Uganda,Gulu,≥0.5,years,67,AL,Sanger,symptomatic,67,missing,S695T,1 (1.5),raw counts,open,1,1.50
5,34551228,2021,2019,Uganda,Northern Uganda,Gulu,≥0.5,years,96,AL,Sanger,symptomatic,96,missing,V555A,1 (1.0),raw counts,open,1,1.00
6,34551228,2021,2019,Uganda,Northern Uganda,Gulu,≥0.5,years,96,AL,Sanger,symptomatic,96,missing,T685P,2 (2.1),raw counts,open,2,2.10
7,34551228,2021,2019,Uganda,Northern Uganda,Gulu,≥0.5,years,96,AL,Sanger,symptomatic,96,missing,L708I,1 (1.0),raw counts,open,1,1.00
8,35703955,2022,missing,DRC,Central DRC,Kinshasha,3-6,years,118,AS+AQ,missing,symptomatic,118,missing,A578S,1/118,raw counts,open,1,0.85
9,35703955,2022,missing,DRC,Central DRC,Kinshasha,3-6,years,118,AS+AQ,missing,symptomatic,118,missing,Q613E,1/118,raw counts,open,1,0.85


In [40]:
# Replace with the actual PMID you're searching for
pmid_to_check = 39136468

# Filter rows where pmid matches
df1_filtered = df1[df1["pmid"] == pmid_to_check]

# Display the result
print(df1_filtered)

        pmid  year_pub year_sample country region/province  \
12  39136468    2024.0   2015-2023  Uganda  Eastern Uganda   
13  39136468    2024.0   2015-2023  Uganda  Eastern Uganda   

                    town participants_age participants_age_unit sample_size  \
12  Tororo, Busia, Mbale             >0.5                 years        1112   
13  Tororo, Busia, Mbale             >0.5                 years        1112   

   first_line_act  ... samples_genotyped k13_validated_mutations  \
12             AL  ...              1112                   C469Y   
13             AL  ...              1112                   A675V   

   type_of_mutation mutation_frequency_(%) where_sequenced  \
12   non-synonymous                      6         missing   
13   non-synonymous                      5         missing   

   other_k13_mutations data_extraction_comments journal_access raw_count  \
12               A578S               raw counts           open         6   
13                   -         

In [41]:
df1_filtered.head()

,pmid,year_pub,year_sample,country,region/province,town,participants_age,participants_age_unit,sample_size,first_line_act,...,samples_genotyped,k13_validated_mutations,type_of_mutation,mutation_frequency_(%),where_sequenced,other_k13_mutations,data_extraction_comments,journal_access,raw_count,percentage_frequency
12,39136468,2024.0,2015-2023,Uganda,Eastern Uganda,"Tororo, Busia, Mbale",>0.5,years,1112,AL,...,1112,C469Y,non-synonymous,6,missing,A578S,raw counts,open,6,0.54
13,39136468,2024.0,2015-2023,Uganda,Eastern Uganda,"Tororo, Busia, Mbale",>0.5,years,1112,AL,...,1112,A675V,non-synonymous,5,missing,-,raw counts,open,5,0.45


In [73]:
df3.head()

,country,town,latitude,longitude
0,Burundi,missing,NaN,NaN
1,DRC,Kinshasa,-4.37500,15.97000
2,DRC,missing,NaN,NaN
3,DRC,"Kabondo,Kapolowe,Rutshuru,Mikalayi,Kimpese",-11.04443,26.94924
4,DRC,Butembo,0.12500,29.29200


In [45]:
# First, drop duplicates from the coordinates file using both keys
df3 = df3[["country", "town", "latitude", "longitude"]].drop_duplicates(subset=["country", "town"])

In [46]:
df4 = df4[["country", "town", "latitude", "longitude"]].drop_duplicates(subset=["country", "town"])

In [47]:
df3.head()

,country,town,latitude,longitude
0,Burundi,missing,NaN,NaN
1,DRC,Kinshasa,-4.37500,15.97000
2,DRC,missing,NaN,NaN
11,DRC,"Kabondo,Kapolowe,Rutshuru,Mikalayi,Kimpese",-11.04443,26.94924
13,DRC,Butembo,0.12500,29.29200


In [48]:
df4.head()

,country,town,latitude,longitude
0,Burundi,missing,NaN,NaN
1,DRC,Kinshasa,-4.37500,15.97000
2,DRC,missing,NaN,NaN
11,DRC,"Kabondo,Kapolowe,Rutshuru,Mikalayi,Kimpese",-11.04443,26.94924
13,DRC,Butembo,0.12500,29.29200


In [49]:
# Merge using both 'country' and 'town' as keys
df_merged = pd.merge(df3, df4, on=["country", "town"], how="left")

In [50]:
df_merged.head()

,country,town,latitude_x,longitude_x,latitude_y,longitude_y
0,Burundi,missing,NaN,NaN,NaN,NaN
1,DRC,Kinshasa,-4.37500,15.97000,-4.37500,15.97000
2,DRC,missing,NaN,NaN,NaN,NaN
3,DRC,"Kabondo,Kapolowe,Rutshuru,Mikalayi,Kimpese",-11.04443,26.94924,-11.04443,26.94924
4,DRC,Butembo,0.12500,29.29200,0.12500,29.29200


In [51]:
# Create clean latitude/longitude by prioritizing non-missing values
df_merged["latitude"] = df_merged["latitude_x"].combine_first(df_merged["latitude_y"])
df_merged["longitude"] = df_merged["longitude_x"].combine_first(df_merged["longitude_y"])

# Drop the duplicate columns
df_merged = df_merged.drop(columns=["latitude_x", "latitude_y", "longitude_x", "longitude_y"])

In [52]:
df_merged.head()

,country,town,latitude,longitude
0,Burundi,missing,NaN,NaN
1,DRC,Kinshasa,-4.37500,15.97000
2,DRC,missing,NaN,NaN
3,DRC,"Kabondo,Kapolowe,Rutshuru,Mikalayi,Kimpese",-11.04443,26.94924
4,DRC,Butembo,0.12500,29.29200


In [53]:
df_merged.to_excel("K13_coordinates_v1.xlsx", index=False)

In [74]:
# Merge using both 'country' and 'town' as keys
final_df = pd.merge(df2, df3, on=["country", "town"], how="left")

In [75]:
final_df

,pmid,year_pub,year_sample,country,region/province,town,participants_age,participants_age_unit,sample_size,first_line_act,...,samples_genotyped,where_sequenced,other_k13_mutations,Frequency,data_extraction_comments,journal_access,raw_count,percentage_frequency,latitude,longitude
0,34551228,2021,2015,Uganda,Northern Uganda,Gulu,≥0.5,years,77,AL,...,77,missing,C532S,2 (2.6),raw counts,open,2,2.60,2.85604,32.43001
1,34551228,2021,2015,Uganda,Northern Uganda,Gulu,≥0.5,years,77,AL,...,77,missing,V661I,1 (1.3),raw counts,open,1,1.30,2.85604,32.43001
2,34551228,2021,2016,Uganda,Northern Uganda,Gulu,≥0.5,years,94,AL,...,94,missing,G595C,1 (1.1),raw counts,open,1,1.10,2.85604,32.43001
3,34551228,2021,2018,Uganda,Northern Uganda,Gulu,≥0.5,years,67,AL,...,67,missing,A578S,1 (1.5),raw counts,open,1,1.50,2.85604,32.43001
4,34551228,2021,2018,Uganda,Northern Uganda,Gulu,≥0.5,years,67,AL,...,67,missing,S695T,1 (1.5),raw counts,open,1,1.50,2.85604,32.43001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1190,32747827,2020,2012–2015,Rwanda,"Kicukiro, Bugesera, Rusizi, Gisagara, Nyagatar...","Masaka,Ruhuha,Bugarama,Kibirizi,Nyarurema,Rukara",1–14,years,954,AL,...,927,"Institut Pasteur, Paris and Rwanda Biomedical ...",E605K,1/927(0.11),raw counts,open,1,0.11,-1.40230,30.17510
1191,32747827,2020,2012–2015,Rwanda,"Kicukiro, Bugesera, Rusizi, Gisagara, Nyagatar...","Masaka,Ruhuha,Bugarama,Kibirizi,Nyarurema,Rukara",1–14,years,954,AL,...,927,"Institut Pasteur, Paris and Rwanda Biomedical ...",A626E,1/927(0.11),raw counts,open,1,0.11,-1.40230,30.17510
1192,32747827,2020,2012–2015,Rwanda,"Kicukiro, Bugesera, Rusizi, Gisagara, Nyagatar...","Masaka,Ruhuha,Bugarama,Kibirizi,Nyarurema,Rukara",1–14,years,954,AL,...,927,"Institut Pasteur, Paris and Rwanda Biomedical ...",V637I,1/927(0.11),raw counts,open,1,0.11,-1.40230,30.17510
1193,32747827,2020,2012–2015,Rwanda,"Kicukiro, Bugesera, Rusizi, Gisagara, Nyagatar...","Masaka,Ruhuha,Bugarama,Kibirizi,Nyarurema,Rukara",1–14,years,954,AL,...,927,"Institut Pasteur, Paris and Rwanda Biomedical ...",E651K,1/927(0.11),raw counts,open,1,0.11,-1.40230,30.17510


In [76]:
final_df.to_excel("K13_non_validated_v3.xlsx", index=False)